## Pretraining the Text Autoencoder on a custom dataset

In [ ]:
!pip install torchinfo
!pip install clip
!pip install evaluate
!pip install rouge_score
!pip install nltk

  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-0.2.0-py3-none-any.whl size=6989 sha256=080cdb97e2b0e08560564caaf793e74a7b0fe6926a30125c88c8b00d247d236e
  Stored in directory: /root/.cache/pip/wheels/6c/fd/54/9d4e15cf829b871199a7cd3597e869a514d1624a0a43076896
Successfully built clip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=dc6afb7eb4c96394b95f9f2424a57e29f2772c072d88952a94897cbf6be8643f
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
# @title Importing the necessary libraries

import torch
import torch.nn as nn
import torch.nn.functional as F
import clip
from bs4 import BeautifulSoup
import re
import matplotlib.pyplot as plt
import numpy as np
import os
from nltk.translate.bleu_score import sentence_bleu
import json
import pandas as pd
from torchinfo import summary
from transformers import CLIPProcessor, CLIPModel, RobertaModel, RobertaTokenizer, GPT2Tokenizer
import evaluate
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import tqdm
from datasets.fingerprint import random
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.transforms.functional as FT
import math
from transformers import BertTokenizer
import gc
import random

from typing import Dict, Any, List, Optional, Tuple
import textwrap
from tqdm import tqdm
from src.utils.helper import parse_gdi_text, save_checkpoint_to_drive, load_checkpoint_from_drive, emb_dim, latent_dim, num_layers, max_seq_len, batch_size, dropout

# **Chapter 1: The data preparation**


---



## 1.1 Loading and saving data

In [ ]:


from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


We need to define a couple of functions to make our life easier. Feel free to tweak those functions:

In [ ]:

emb_dim = emb_dim
latent_dim = latent_dim
num_layers = num_layers
max_seq_len = max_seq_len
batch_size = batch_size
dropout = dropout

from transformers import get_cosine_schedule_with_warmup

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

nltk.download("punkt")
nltk.download("wordnet")
nltk.download('punkt_tab')


Now we load dataset from HuggingFace:

In [ ]:
# @title Loading the dataset
from datasets import load_dataset

train_dataset = load_dataset("qwedsacf/story-generation")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.94k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/327M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/331M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/115M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3552 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/626 [00:00<?, ? examples/s]

In [ ]:
# @title CoT improvements toggles (added)



USE_FRAME_AWARE_GROUNDING = True
USE_CONTRASTIVE_ROI = True
USE_ENTITY_POOLING = True
USE_COT_TEXT = True


CONTRASTIVE_TAU = 0.07

In [ ]:
from src.utils.helper import TextDataset

enc_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
dec_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
text_dataset = TextDataset(
    train_dataset['train'], enc_tokenizer, dec_tokenizer,
    max_len=max_seq_len, max_target_len=256, text_field="story",
)
text_dataloader = DataLoader(text_dataset, batch_size=batch_size, shuffle=True)
for batch in text_dataloader:
    print(batch["input_ids"].shape, batch["attention_mask"].shape, batch["labels"].shape)
    break

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:

for batch in text_dataloader:
    print(batch["input_ids"].shape, batch["attention_mask"].shape)
    break

torch.Size([16, 128]) torch.Size([16, 128])


In [ ]:
print(train_dataset[0].keys())

dict_keys(['story_id', 'images', 'frame_count', 'chain_of_thought', 'story'])


## 3.1 Initialization and setup

In [ ]:
# @title Initializing the Roberta model
from src.models.text_autoencoder import RobertaEncoder, TransformerDecoder, Seq2Seq, SinusoidalPositionalEncoding
torch.cuda.empty_cache()
gc.collect()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dec_tokenizer.pad_token = dec_tokenizer.eos_token      # GPT-2 has no pad token

text_autoencoder = Seq2Seq(
    encoder_name="roberta-base",
    decoder_name="gpt2",
    unfreeze_encoder_layers=6,
    enc_tokenizer=enc_tokenizer,
    dec_tokenizer=dec_tokenizer,
).to(device)

trainable = sum(p.numel() for p in text_autoencoder.parameters() if p.requires_grad)
total     = sum(p.numel() for p in text_autoencoder.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Total params:     {total:,}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 123668736
Total params: 219962880


In [ ]:
enc_len = min(max_seq_len, 512)        # RoBERTa caps at 512
dec_len = 256                          # target length (match your max_target_len)

dummy_input  = torch.randint(0, enc_tokenizer.vocab_size, (batch_size, enc_len)).to(device)
dummy_mask   = torch.ones((batch_size, enc_len), dtype=torch.long).to(device)   # all valid
dummy_labels = torch.randint(0, dec_tokenizer.vocab_size, (batch_size, dec_len)).to(device)

print("===== RoBERTa2GPT2 Seq2Seq Summary =====")
summary(
    text_autoencoder,
    input_data=(dummy_input, dummy_mask, dummy_labels),   # input_ids, attention_mask, labels
    col_names=["input_size", "output_size", "num_params", "trainable"],
    depth=3,
)

===== Roberta Text Autoencoder Summary =====


Layer (type:depth-idx)                                            Input Shape               Output Shape              Param #                   Trainable
Seq2Seq                                                           [16, 120]                 [16, 120, 50265]          --                        Partial
├─RobertaEncoder: 1-1                                             --                        [16, 120, 768]            --                        Partial
│    └─RobertaModel: 2-1                                          --                        [16, 768]                 --                        Partial
│    │    └─RobertaEmbeddings: 3-1                                --                        [16, 120, 768]            (39,000,576)              False
│    │    └─RobertaEncoder: 3-2                                   [16, 120, 768]            [16, 120, 768]            85,054,464                Partial
│    │    └─RobertaPooler: 3-3                                    [16, 120, 768]        

## 3.2 Training loops

In [ ]:
from src.training.train_text import (
    ContrastiveDiversityLoss,
    EarlyStopping,
    build_optimizer,
    build_scheduler,
    train_text_autoencoder,
)

# =========================================================
# FIXED EVALUATION PROMPTS
# =========================================================
EVAL_PROMPT = (
    "The water was clear and bright. After days in the shade, sunlight now "
    "cut through the rough surface, sending shimmering rays dancing across "
    "the rocky bed of the river, and illuminating the patches of bright green "
    "algae that carpeted the rocks of deeper, slower pools.\n\n"
    "The fish gained speed, then hurled herself above the water line, over "
    "the rocks that created a minor fall in the river."
)

EVAL_REFERENCE = (
    "She had, not so long ago, lived in the deep sea. Her lithe silver body "
    "pulling saltwater through her gills as she explored the dark depths, "
    "feeding on its bounty.\n\nNow, she fought her way upstream in the clear, "
    "saltless water."
)

DIVERSITY_PROMPTS = {
    "river_fish":   "The fish swam against the powerful river current beneath bright morning sunlight.",
    "space":        "The abandoned space station drifted silently through the darkness of deep space.",
    "mansion":      "Thunder rolled over the old mansion as footsteps echoed through the empty halls.",
    "forest":       "Morning mist clung to the ancient trees as birds began their first tentative calls.",
    "desert":       "The sun hammered the cracked earth, and nothing moved for miles in any direction.",
}

# =========================================================
# TRAINING CONFIG
# =========================================================
N_EPOCHS        = 5
LR_DECODER      = 1e-4    # decoder + fresh layers – higher LR fine
LR_ENCODER      = 1e-5    # pretrained RoBERTa – 10× lower to preserve weights
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 1.0
LABEL_SMOOTHING = 0.10    # raised from 0.05 to discourage overconfident copying
W_CE            = 1.0
W_DIVERSITY     = 0.15    # weight for contrastive diversity loss

checkpoint_filename = "text_autoencoder.pth"

optimizer = build_optimizer(text_autoencoder, lr_encoder=1e-5, lr_decoder=1e-4, weight_decay=0.01)
scheduler = build_scheduler(optimizer, len(text_dataloader), N_EPOCHS)


history = train_text_autoencoder(
    model=text_autoencoder,
    dataloader=text_dataloader,
    enc_tokenizer=enc_tokenizer,
    dec_tokenizer=dec_tokenizer,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    n_epochs=N_EPOCHS,
    checkpoint_filename=checkpoint_filename,
    eval_prompt=EVAL_PROMPT,
    eval_reference=EVAL_REFERENCE,
    diversity_prompts=DIVERSITY_PROMPTS,
    max_grad_norm=1.0,
)